# Loaded and initialised

In [ ]:
import sys
sys.path.append('..')
import torch
import torch.nn as nn
from tqdm import tqdm
from models import GenStudent, GenClassifier, vit_parameters, classifier_layels_list
from utils import save_model
from utils.dataset import get_datasets, image_transform
from utils.channels import Channels

device = 'cuda:1'
student = GenStudent(vit_parameters, device)
classifier = GenClassifier(512, 100, classifier_layels_list, device)
student.load_state_dict(torch.load('../results/weights/e2e_train_student/e2e_student_model.pt', map_location=device))
classifier.load_state_dict(torch.load('../results/weights/e2e_train_student/e2e_classifier.pt', map_location=device))
train_loader, test_loader = get_datasets(image_transform, 500, 'cifar100')

# End-to-end training of student models

In [ ]:
student.train()
classifier.train()

optimizer_stu = torch.optim.Adam(student.parameters(), lr=1e-3)
optimizer_cls = torch.optim.Adam(classifier.parameters(), lr=1e-3)

criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    for i, (images, target) in enumerate(train_loader):
        images, target = images.to(device), target.to(device)

        optimizer_stu.zero_grad()
        optimizer_cls.zero_grad()

        student_output = student(images)
        classifier_output = classifier(student_output)

        loss = criterion(classifier_output, target)
        acc = (classifier_output.argmax(1) == target).float().mean()
        loss.backward()

        optimizer_stu.step()
        optimizer_cls.step()

        if i % 100 == 0:
            print(f'epoch: {epoch}, step: {i}, loss: {loss.item():.4f}, acc: {acc.item():.4f}')

# Tested at different SNRs

In [ ]:
student.eval()
classifier.eval()
channels = Channels(device)
criterion = nn.CrossEntropyLoss()

ACC_with_SNR = []
for snr in range(-10, 25, 2):
    test_loader_tqdm = tqdm(test_loader, desc='Testing on SNR:'+str(snr), ncols=120)
    Length = 0
    acc = 0
    for i, (images, labels) in enumerate(test_loader):
        images, labels = images.to(device), labels.to(device)

        with torch.no_grad():

            features = student(images)
            features_awgn = channels.AWGN(features, snr)
            logits = classifier(features)

            preds = logits.argmax(dim=1)
            acc += (preds == labels).float().sum()

        Length = (i+1)*len(labels)

        test_loader.set_postfix(Acc = acc.item() / Length)
        test_loader.update()
    ACC_with_SNR.append(acc.item() / Length)

print(ACC_with_SNR)

# Freeze the student model and train a new classification network

In [ ]:
classifier_new = GenClassifier(512, 100, classifier_layels_list, device)
student.eval()
classifier_new.train()

optimizer_cls_new = torch.optim.Adam(classifier_new.parameters(), lr=1e-4)

for epoch in range(3):
    for i, (images, target) in enumerate(train_loader):
        images, target = images.to(device), target.to(device)

        optimizer_cls_new.zero_grad()

        student_output = student(images)
        classifier_output = classifier_new(student_output)

        loss = criterion(classifier_output, target)
        acc = (classifier_output.argmax(1) == target).float().mean()
        loss.backward()

        optimizer_cls_new.step()

        if i % 100 == 0:
            print(f'epoch: {epoch}, step: {i}, loss: {loss.item():.4f}, acc: {acc.item():.4f}')

# Save model weights

In [ ]:
# save_model(student, '../results/weights/e2e_train_student/student.pt')
# save_model(classifier, '../results/weights/e2e_train_student/classifier.pt')
save_model(classifier_new, '../results/weights/e2e_train_student/e2e_classifier_new.pt')